# p-adic Data Science — Complete Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mircus/padics/blob/master/notebooks/00_tutorial.ipynb)

A hands-on tour of **padic-ds**: p-adic arithmetic, Hensel lifting, ultrametric distances, hierarchical clustering and k-NN classification.

**Contents:**
1. Installation & imports
2. p-adic numbers: construction & arithmetic
3. p-adic absolute value & the ultrametric
4. p-adic balls
5. Hensel lifting — finding roots in Zₚ
6. Pairwise distance matrix
7. Hierarchical clustering (dendrogram)
8. p-adic k-NN classifier
9. MDS visualisation

In [ ]:
# Run this cell to install padic-ds (needed on Colab or a fresh environment)
import importlib, subprocess, sys
if importlib.util.find_spec("padic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/Mircus/padics.git"])
    print("padic-ds installed")
else:
    print("padic already available")

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from sklearn.manifold import MDS
from sklearn.model_selection import train_test_split

from padic import (
    QpContext, Qp, QpBall,
    padic_abs, padic_dist, pairwise_padic_dist,
    hensel_lift_simple, PadicKNNClassifier, embed_float_array,
    digits_p_adic,
)
print("padic-ds loaded successfully ✓")

## 1 · Constructing p-adic numbers

In [ ]:
ctx = QpContext(p=3, prec=8)          # Q_3 with 8-digit precision

x = Qp.from_int(ctx, 18)             # 18 = 2 · 3²
y = Qp.from_rational(ctx, 1, 3)      # 1/3 = 3^{-1}
z = Qp.from_int(ctx, 1)

print(f"18    → {x}")
print(f"1/3   → {y}")
print(f"  |18|_3   = {padic_abs(x):.8f}  (= 3^(-2) = 1/9)")
print(f"  |1/3|_3  = {padic_abs(y):.8f}  (= 3^(+1) = 3)")
print(f"  |1|_3    = {padic_abs(z):.8f}  (= 3^(0)  = 1)")

## 2 · p-adic distance & ultrametric property

In [ ]:
ctx5 = QpContext(p=5, prec=10)
a = Qp.from_int(ctx5, 25)    # 5²
b = Qp.from_int(ctx5, 125)   # 5³
c = Qp.from_int(ctx5, 150)   # 6 · 5²

dab = padic_dist(a, b)
dbc = padic_dist(b, c)
dac = padic_dist(a, c)

print(f"dist(25, 125) = {dab:.6f}")
print(f"dist(125,150) = {dbc:.6f}")
print(f"dist(25, 150) = {dac:.6f}")
print()
print(f"Ultrametric: dist(a,c) ≤ max(dist(a,b), dist(b,c))")
print(f"  {dac:.6f} ≤ max({dab:.6f}, {dbc:.6f}) = {max(dab,dbc):.6f}  {'✓' if dac <= max(dab,dbc)+1e-12 else '✗'}")

## 3 · p-adic balls

In [ ]:
ctx5 = QpContext(p=5, prec=6)
center = Qp.from_int(ctx5, 0)
ball = QpBall(center, 1.0/25)   # radius 5^{-2} = 0.04

print("Ball B(0, 1/25) in Q_5:")
print(f"{'n':>5}  {'|n|_5':>8}  {'in ball?':>10}")
print("-" * 30)
for n in [0, 5, 10, 25, 50, 1, 2, 3]:
    pt = Qp.from_int(ctx5, n)
    print(f"{n:>5}  {padic_abs(pt):>8.5f}  {str(ball.contains(pt)):>10}")

## 4 · Hensel lifting — √2 in Z₇

In [ ]:
# f(X) = X²-2,  f'(X) = 2X
# 3² = 9 ≡ 2 mod 7,  so a₀ = 3 is a starting root mod 7
ctx7 = QpContext(p=7, prec=12)
root = hensel_lift_simple(ctx7, lambda a: a*a-2, lambda a: 2*a,
                           a0_mod_p=3, target_prec=12)
print(f"√2 in Z_7 (12 digits):")
print(f"  {root}")
digs = digits_p_adic(root, 12)
print(f"  Base-7 digits: {digs}")
print(f"  First digit (should be 3, since 3²≡2 mod 7): {digs[0]}")

## 5 · Pairwise distance matrix

In [ ]:
ctx3 = QpContext(p=3, prec=8)
nums   = [1, 2, 3, 9, 27, 81]
points = [Qp.from_int(ctx3, n) for n in nums]
D      = pairwise_padic_dist(points)

print("3-adic pairwise distance matrix:")
print("     " + "  ".join(f"{n:>6}" for n in nums))
for i, row in enumerate(D):
    print(f"{nums[i]:4} [{' '.join(f'{v:>6.4f}' for v in row)}]")

## 6 · Hierarchical clustering

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.imshow(D, cmap="YlOrRd_r", vmin=0)
ax.set_xticks(range(len(nums))); ax.set_yticks(range(len(nums)))
ax.set_xticklabels(nums); ax.set_yticklabels(nums)
plt.colorbar(im, ax=ax, label="3-adic distance")
ax.set_title("3-adic distance matrix")

ax = axes[1]
Z = linkage(squareform(D, checks=False), method="complete")
dendrogram(Z, labels=[str(n) for n in nums], ax=ax, color_threshold=0.15)
ax.set_title("Hierarchical clustering (3-adic)")
ax.set_ylabel("3-adic distance")

plt.tight_layout()
plt.savefig("ultrametric_cluster.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved ultrametric_cluster.png")

## 7 · p-adic k-NN: classify by 3-adic valuation

In [ ]:
# Label each integer by its 3-adic valuation (capped at 2):
# v_3(n) = 0  → class 0   (not divisible by 3)
# v_3(n) = 1  → class 1   (divisible by 3 but not 9)
# v_3(n) ≥ 2  → class 2   (divisible by 9)
def valuation_class(n):
    v = 0
    while n % 3 == 0:
        n //= 3; v += 1
    return min(v, 2)

np.random.seed(42)
integers = np.random.randint(1, 500, size=150)
y        = np.array([valuation_class(n) for n in integers])

# embed integers as 1-D Qp vectors
ctx3   = QpContext(p=3, prec=10)
X_qp   = embed_float_array(integers.reshape(-1, 1), ctx3, scale=1)

X_tr, X_te, y_tr, y_te = train_test_split(X_qp, y, test_size=0.25, random_state=0)
print(f"Train: {len(X_tr)}  Test: {len(X_te)}")
print(f"Class counts (train): {np.bincount(y_tr)}")
print()
for k in [3, 5, 7]:
    clf = PadicKNNClassifier(ctx=ctx3, k=k)
    clf.fit(X_tr, y_tr)
    acc = clf.score(X_te, y_te)
    print(f"k={k}: accuracy = {acc:.2%}")

## 8 · MDS: 3-adic topology of integers 1–27

In [ ]:
vals = list(range(1, 28))
pts  = [Qp.from_int(ctx3, n) for n in vals]
D28  = pairwise_padic_dist(pts)

xy = MDS(n_components=2, dissimilarity="precomputed", random_state=42).fit_transform(D28)

def color(n):
    if n % 9 == 0:  return "#e41a1c"   # red  : div by 9
    if n % 3 == 0:  return "#ff7f00"   # orange: div by 3 not 9
    return "#377eb8"                    # blue : not div by 3

fig, ax = plt.subplots(figsize=(10, 7))
for (xi, yi), n in zip(xy, vals):
    ax.scatter(xi, yi, color=color(n), s=100, zorder=3)
    ax.annotate(str(n), (xi, yi), xytext=(4, 4), textcoords="offset points", fontsize=9)

ax.legend(handles=[
    mpatches.Patch(color="#e41a1c", label="div by 9  (v₃≥2)"),
    mpatches.Patch(color="#ff7f00", label="div by 3, not 9  (v₃=1)"),
    mpatches.Patch(color="#377eb8", label="not div by 3  (v₃=0)"),
])
ax.set_title("MDS of integers 1–27 in 3-adic metric\n(numbers with same 3-adic valuation cluster together)")
ax.set_xlabel("MDS dim 1"); ax.set_ylabel("MDS dim 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("padic_mds.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved padic_mds.png")